In [4]:
import keras
import seaborn as sns
import matplotlib.pyplot as plt
from keras.models import Sequential
import PIL
import os 
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from keras.layers import Conv2D,Flatten,Dense,Dropout,BatchNormalization,MaxPooling2D
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from keras.callbacks import EarlyStopping,ModelCheckpoint
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from tqdm import tqdm
from imblearn.over_sampling import SMOTE

In [5]:
non_demented = []
very_mild_demented = []
mild_demented = []
moderate_demented = []

for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Non Demented'):
    for filename in filenames:
        non_demented.append(os.path.join(dirname, filename))
        
for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Very mild Dementia'):
    for filename in filenames:
        very_mild_demented.append(os.path.join(dirname, filename))
        
for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Mild Dementia'):
    for filename in filenames:
        mild_demented.append(os.path.join(dirname, filename))
        
for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Moderate Dementia'):
    for filename in filenames:
        moderate_demented.append(os.path.join(dirname, filename))

KeyboardInterrupt: 

In [13]:


MAX_PER_CLASS = 250

import random

random.shuffle(non_demented)
random.shuffle(very_mild_demented)
random.shuffle(mild_demented)
random.shuffle(moderate_demented)

non_demented = non_demented[:MAX_PER_CLASS]
very_mild_demented = very_mild_demented[:MAX_PER_CLASS]
mild_demented = mild_demented[:MAX_PER_CLASS]
moderate_demented = moderate_demented[:MAX_PER_CLASS]

print("Non Demented:", len(non_demented))
print("Very Mild Demented:", len(very_mild_demented))
print("Mild Demented:", len(mild_demented))
print("Moderate Demented:", len(moderate_demented))

print("Total Images:",
      len(non_demented)
    + len(very_mild_demented)
    + len(mild_demented)
    + len(moderate_demented))

Non Demented: 250
Very Mild Demented: 250
Mild Demented: 250
Moderate Demented: 250
Total Images: 1000


In [ ]:

# ============================================================
# IMPORTS
# ============================================================

import os
import random
import numpy as np
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# ============================================================
# LOAD IMAGE PATHS
# ============================================================

non_demented = []
very_mild_demented = []
mild_demented = []
moderate_demented = []

for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Non Demented'):
    for filename in filenames:
        non_demented.append(os.path.join(dirname, filename))

for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Very mild Dementia'):
    for filename in filenames:
        very_mild_demented.append(os.path.join(dirname, filename))

for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Mild Dementia'):
    for filename in filenames:
        mild_demented.append(os.path.join(dirname, filename))

for dirname, _, filenames in os.walk('/kaggle/input/imagesoasis/Data/Moderate Dementia'):
    for filename in filenames:
        moderate_demented.append(os.path.join(dirname, filename))

# ============================================================
# KEEP ONLY 250 PER CLASS
# ============================================================

MAX_PER_CLASS = 250

random.shuffle(non_demented)
random.shuffle(very_mild_demented)
random.shuffle(mild_demented)
random.shuffle(moderate_demented)

non_demented = non_demented[:MAX_PER_CLASS]
very_mild_demented = very_mild_demented[:MAX_PER_CLASS]
mild_demented = mild_demented[:MAX_PER_CLASS]
moderate_demented = moderate_demented[:MAX_PER_CLASS]

print("Total Images:",
      len(non_demented)
    + len(very_mild_demented)
    + len(mild_demented)
    + len(moderate_demented))

# ============================================================
# LOAD IMAGES
# ============================================================

X = []

all_images = (
    non_demented +
    very_mild_demented +
    mild_demented +
    moderate_demented
)

for path in all_images:

    img = Image.open(path).convert("RGB")
    img = img.resize((128,128))

    X.append(np.array(img))

X = np.array(X, dtype=np.float32)

print("X Shape:", X.shape)

# EXPECT:
# (1000,128,128,3)

# ============================================================
# MOBILENET FEATURE EXTRACTOR
# ============================================================

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(128,128,3)
)

embedding_model = Model(
    inputs=base_model.input,
    outputs=GlobalAveragePooling2D()(base_model.output)
)

# ============================================================
# PREPROCESS
# ============================================================

X_mobile = preprocess_input(X.copy())

# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

embeddings = embedding_model.predict(
    X_mobile,
    batch_size=32,
    verbose=1
)

print("Embeddings Shape:", embeddings.shape)

# EXPECT:
# (1000,1280)

# ============================================================
# PCA
# ============================================================

pca = PCA(n_components=128)

embeddings_128 = pca.fit_transform(
    embeddings
)

print("Reduced Shape:", embeddings_128.shape)

# EXPECT:
# (1000,128)

# ============================================================
# SAVE FOR TEAM
# ============================================================

np.save(
    "embeddings.npy",
    embeddings_128
)

print("embeddings.npy saved")

# ============================================================
# SIMPLE BRAIN TWIN FINDER
# ============================================================

query_id = 0

scores = cosine_similarity(
    [embeddings_128[query_id]],
    embeddings_128
)

top5 = np.argsort(scores[0])[::-1][1:6]

print("\nQuery Patient:", query_id)
print("Top 5 Similar Patients:")
print(top5)



In [12]:
from sklearn.decomposition import PCA

X_flat = X.reshape(1000, -1)

print(X_flat.shape)

(1000, 49152)


In [15]:
pca = PCA(n_components=8)

embeddings = pca.fit_transform(X_flat)

print(embeddings.shape)

(1000, 8)


In [21]:
np.save("embeddings.npy", embeddings)

In [18]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embeddings = np.load("embeddings.npy")

query_id = 0

scores = cosine_similarity(
    [embeddings[query_id]],
    embeddings
)

top5 = np.argsort(scores[0])[::-1][1:6]

print(top5)

[155 334 330 322 377]


In [19]:
labels = (
    ["Non Demented"] * 250 +
    ["Very Mild Dementia"] * 250 +
    ["Mild Dementia"] * 250 +
    ["Moderate Dementia"] * 250
)

print("Query Patient Label:")
print(labels[0])

print("\nRetrieved Patients:")

for idx in [155,334,330,322,377]:
    print(idx, labels[idx])

Query Patient Label:
Non Demented

Retrieved Patients:
155 Non Demented
334 Very Mild Dementia
330 Very Mild Dementia
322 Very Mild Dementia
377 Very Mild Dementia


In [20]:
for query_id in [50,150,300,500,700]:
    scores = cosine_similarity(
        [embeddings[query_id]],
        embeddings
    )

    top5 = np.argsort(scores[0])[::-1][1:6]

    print("\nQuery:", query_id)
    print("Label:", labels[query_id])

    for idx in top5:
        print(idx, labels[idx])


Query: 50
Label: Non Demented
38 Non Demented
489 Very Mild Dementia
470 Very Mild Dementia
100 Non Demented
387 Very Mild Dementia

Query: 150
Label: Non Demented
20 Non Demented
483 Very Mild Dementia
461 Very Mild Dementia
94 Non Demented
2 Non Demented

Query: 300
Label: Very Mild Dementia
284 Very Mild Dementia
69 Non Demented
427 Very Mild Dementia
343 Very Mild Dementia
366 Very Mild Dementia

Query: 500
Label: Mild Dementia
744 Mild Dementia
737 Mild Dementia
662 Mild Dementia
709 Mild Dementia
846 Moderate Dementia

Query: 700
Label: Mild Dementia
315 Very Mild Dementia
545 Mild Dementia
672 Mild Dementia
685 Mild Dementia
657 Mild Dementia


In [22]:
from sklearn.decomposition import PCA
import pandas as pd

# embeddings = your feature matrix

pca = PCA(n_components=8)

brain_signatures = pca.fit_transform(embeddings)

print(brain_signatures.shape)

(1000, 8)


In [25]:
from sklearn.decomposition import PCA
import pandas as pd

# Create 8-dimensional signatures
pca = PCA(n_components=8)
brain_signatures = pca.fit_transform(embeddings)

# Create dataframe
df = pd.DataFrame(
    brain_signatures,
    columns=["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8"]
)

# Save CSV
df.to_csv("/kaggle/working/brain_signatures_8d.csv", index=False)

print(df.shape)
print("Saved!")

(1000, 8)
Saved!


In [26]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.read_csv("brain_signatures_8d.csv")

scaler = StandardScaler()

scaled = scaler.fit_transform(df)

scaled_df = pd.DataFrame(
    scaled,
    columns=df.columns
)

scaled_df.to_csv(
    "brain_signatures_8d_scaled.csv",
    index=False
)

In [3]:
print(len(non_demented))
print(len(very_mild_demented))
print(len(mild_demented))
print(len(moderate_demented))

NameError: name 'non_demented' is not defined